In [1]:
import collections
import re

from d2l import torch as d2l

In [2]:
class TimeMachine(d2l.DataModule):
    """The Time Machine dataset."""

    def _download(self) -> str:
        filename = d2l.download(
            d2l.DATA_URL + "timemachine.txt",
            self.root,
            "090b5e7e70c295757f55df93cb0a180b9691891a",
        )

        with open(filename, encoding="utf-8") as file:
            return file.read()

    def _preprocess(self, text: str) -> str:
        return re.sub(
            r"[^A-Za-z]+",
            " ",
            text,
        ).lower()

    def _tokenize(self, text: str) -> list[str]:
        return list(text)

In [3]:
# Preprocessing & Tokenization

data = TimeMachine()
raw_text = data._download()
text = data._preprocess(raw_text)
tokens: list[str] = data._tokenize(text)

print(repr(text[:60]))
print("First 10 tokens:", tokens[:10])
print("Token count:", len(tokens))

assert len(tokens) == len(text)
assert "".join(tokens) == text

'the time machine by h g wells i the time traveller for so it'
First 10 tokens: ['t', 'h', 'e', ' ', 't', 'i', 'm', 'e', ' ', 'm']
Token count: 173428


In [4]:
class Vocab:
    """Vocabulary for text."""

    def __init__(
        self,
        tokens: list[str] | list[list[str]] | None = None,
        min_freq: int = 0,
        reserved_tokens: list[str] | None = None,
    ) -> None:
        if tokens is None:
            tokens = []

        if reserved_tokens is None:
            reserved_tokens = []

        flat_tokens: list[str] = []

        for token_or_line in tokens:
            if isinstance(token_or_line, str):
                flat_tokens.append(token_or_line)
            else:
                flat_tokens.extend(token_or_line)

        counter = collections.Counter(flat_tokens)

        self.token_freqs: list[tuple[str, int]] = sorted(
            counter.items(),
            key=lambda item: item[1],
            reverse=True,
        )

        frequent_tokens = [
            token
            for token, frequency in self.token_freqs
            if frequency >= min_freq
        ]

        self.idx_to_token: list[str] = sorted(
            set([
                "<unk>",
                *reserved_tokens,
                *frequent_tokens,
            ])
        )

        self.token_to_idx: dict[str, int] = {
            token: index
            for index, token in enumerate(self.idx_to_token)
        }

    def __len__(self) -> int:
        return len(self.idx_to_token)

    def __getitem__(
        self,
        tokens: str | list[str] | tuple[str, ...],
    ) -> int | list[int]:
        if isinstance(tokens, str):
            return self.token_to_idx.get(tokens, self.unk)

        return [
            self.token_to_idx.get(token, self.unk)
            for token in tokens
        ]

    def to_tokens(
        self,
        indices: int | list[int] | tuple[int, ...],
    ) -> str | list[str]:
        if isinstance(indices, int):
            return self.idx_to_token[indices]

        return [
            self.idx_to_token[int(index)]
            for index in indices
        ]

    @property
    def unk(self) -> int:
        return self.token_to_idx["<unk>"]

In [5]:
# Numericalize

vocab = Vocab(tokens)
indices = vocab[tokens[:10]]

if not isinstance(indices, list):
    raise TypeError("A token sequence must map to a list of indices.")

reconstructed_tokens = vocab.to_tokens(indices)

if not isinstance(reconstructed_tokens, list):
    raise TypeError("An index sequence must map to a list of tokens.")

print("Vocabulary size:", len(vocab))
print("indices:", indices)
print("tokens:", reconstructed_tokens)
print("unknown index:", vocab.unk)

assert reconstructed_tokens == tokens[:10]
assert vocab["?"] == vocab.unk

Vocabulary size: 28
indices: [21, 9, 6, 0, 21, 10, 14, 6, 0, 14]
tokens: ['t', 'h', 'e', ' ', 't', 'i', 'm', 'e', ' ', 'm']
unknown index: 1


In [6]:
def time_machine_build(
    self: TimeMachine,
    raw_text: str,
    vocab: Vocab | None = None,
) -> tuple[list[int], Vocab]:
    """Preprocess, tokenize, and numericalize raw text."""

    processed_text = self._preprocess(raw_text)
    tokens = self._tokenize(processed_text)

    if vocab is None:
        vocab = Vocab(tokens)

    corpus: list[int] = []

    for token in tokens:
        token_index = vocab[token]

        if not isinstance(token_index, int):
            raise TypeError(
                "Each token must map to one integer index."
            )

        corpus.append(token_index)

    return corpus, vocab


TimeMachine.build = time_machine_build

In [7]:
corpus, vocab = data.build(raw_text)
first_tokens = vocab.to_tokens(corpus[:20])

if not isinstance(first_tokens, list):
    raise TypeError("An index sequence must map to a list of tokens.")

print("Corpus length:", len(corpus))
print("Vocabulary size:", len(vocab))
print("First 20 indices:", corpus[:20])
print("First 20 tokens:", first_tokens)

assert len(corpus) == len(tokens)
assert first_tokens == tokens[:20]

Corpus length: 173428
Vocabulary size: 28
First 20 indices: [21, 9, 6, 0, 21, 10, 14, 6, 0, 14, 2, 4, 9, 10, 15, 6, 0, 3, 26, 0]
First 20 tokens: ['t', 'h', 'e', ' ', 't', 'i', 'm', 'e', ' ', 'm', 'a', 'c', 'h', 'i', 'n', 'e', ' ', 'b', 'y', ' ']
